# DEAI – Data Warehouse voor bestelgedrag Great Outdoors

Dit notebook bouwt een SQLite **Data Warehouse (DWH)** voor het knelpunt:

> Er is onvoldoende inzicht in het bestelgedrag van klanten.

Het DWH maakt analyses mogelijk zoals:
- omzet per periode;
- omzet per regio;
- omzet per klant / klantsegment;
- meest verkochte producten;
- productcombinaties die vaak samen besteld worden;
- topklanten op basis van omzet.

## Bronnen
- `CRM-data.sqlite`
- `GO_SALES-data.sqlite`
- `GO_STAFF-data.sqlite`
- `INVENTORY_LEVELS-data.csv`
- `PRODUCT_FORECAST-data.csv`
- `SALES_TARGET-data.csv`

## Hoofdstructuur
Het DWH gebruikt een sterschema met als kern:

- `fact_order_sales`  
  Granulariteit: **één regel per orderregel / orderdetail**

Daaromheen staan dimensies:
- `dim_date`
- `dim_product`
- `dim_customer`
- `dim_region`
- `dim_sales_staff`
- `dim_order_method`


## 1. Imports en paden


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import logging


In [2]:
# Zet de databestanden in dezelfde map als dit notebook.
BASE_DIR = Path.cwd()

CRM_DB = BASE_DIR / "CRM-data.sqlite"
SALES_DB = BASE_DIR / "GO_SALES-data.sqlite"
STAFF_DB = BASE_DIR / "GO_STAFF-data.sqlite"

INVENTORY_CSV = BASE_DIR / "INVENTORY_LEVELS-data.csv"
FORECAST_CSV = BASE_DIR / "PRODUCT_FORECAST-data.csv"
TARGET_CSV = BASE_DIR / "SALES_TARGET-data.csv"

DWH_DB = BASE_DIR / "GO_DWH.db"

LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
DWH_LOG_PATH = LOG_DIR / "go_dwh_etl.log"

print("BASE_DIR:", BASE_DIR)
print("DWH_DB:", DWH_DB)
print("LOG:", DWH_LOG_PATH)


BASE_DIR: /Users/cmok/Documents/GitHub/DEAI-SE4/Great_Outdoors
DWH_DB: /Users/cmok/Documents/GitHub/DEAI-SE4/Great_Outdoors/GO_DWH.db
LOG: /Users/cmok/Documents/GitHub/DEAI-SE4/Great_Outdoors/logs/go_dwh_etl.log


## 2. Logging instellen


In [3]:
logger = logging.getLogger("go_dwh_etl")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s|%(levelname)s|%(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler = logging.FileHandler(DWH_LOG_PATH, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.propagate = False


def log_event(level, process_step, table_name="", action="", row_count="", details=""):
    message = f"{process_step}|{table_name}|{action}|{row_count}|{details}"
    if level == "ERROR":
        logger.error(message)
    elif level == "WARNING":
        logger.warning(message)
    else:
        logger.info(message)


def reset_dwh_log():
    if logger.hasHandlers():
        for handler in list(logger.handlers):
            handler.close()
            logger.removeHandler(handler)

    if DWH_LOG_PATH.exists():
        DWH_LOG_PATH.unlink()

    new_handler = logging.FileHandler(DWH_LOG_PATH, encoding="utf-8")
    new_handler.setLevel(logging.INFO)
    new_handler.setFormatter(formatter)
    logger.addHandler(new_handler)

    print(f"Logbestand gereset: {DWH_LOG_PATH}")


## 3. Bestandcontrole


In [4]:
def check_files():
    files = [CRM_DB, SALES_DB, STAFF_DB, INVENTORY_CSV, FORECAST_CSV, TARGET_CSV]
    missing = []

    for path in files:
        if path.exists():
            log_event("INFO", "FILE_CHECK", path.name, "FOUND", "", str(path))
            print("Gevonden:", path.name)
        else:
            log_event("ERROR", "FILE_CHECK", path.name, "MISSING", "", str(path))
            missing.append(path)

    if missing:
        raise FileNotFoundError("Ontbrekende bestanden: " + ", ".join(str(p) for p in missing))

    print("Alle bronbestanden zijn gevonden.")


check_files()


Gevonden: CRM-data.sqlite
Gevonden: GO_SALES-data.sqlite
Gevonden: GO_STAFF-data.sqlite
Gevonden: INVENTORY_LEVELS-data.csv
Gevonden: PRODUCT_FORECAST-data.csv
Gevonden: SALES_TARGET-data.csv
Alle bronbestanden zijn gevonden.


## 4. DWH-schema


In [5]:
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL UNIQUE,
    day_of_month INTEGER,
    month_num INTEGER,
    month_name TEXT,
    quarter_num INTEGER,
    year_num INTEGER,
    week_num INTEGER,
    day_name TEXT,
    is_weekend INTEGER
);

CREATE TABLE IF NOT EXISTS dim_product (
    product_key INTEGER PRIMARY KEY AUTOINCREMENT,
    product_bk INTEGER NOT NULL UNIQUE,
    product_name TEXT,
    product_type_code INTEGER,
    product_type TEXT,
    product_line_code INTEGER,
    product_line TEXT,
    introduction_date TEXT,
    production_cost REAL,
    margin REAL
);

CREATE TABLE IF NOT EXISTS dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_site_bk INTEGER NOT NULL UNIQUE,
    customer_bk INTEGER,
    customer_master_bk INTEGER,
    retailer_name TEXT,
    company_name TEXT,
    customer_type TEXT,
    segment_code INTEGER,
    segment_name TEXT,
    city TEXT,
    region TEXT,
    postal_zone TEXT,
    country_code INTEGER,
    country TEXT,
    territory_name TEXT,
    active_indicator INTEGER
);

CREATE TABLE IF NOT EXISTS dim_region (
    region_key INTEGER PRIMARY KEY AUTOINCREMENT,
    region_bk TEXT NOT NULL UNIQUE,
    city TEXT,
    region TEXT,
    country_code INTEGER,
    country TEXT,
    territory_name TEXT
);

CREATE TABLE IF NOT EXISTS dim_sales_staff (
    sales_staff_key INTEGER PRIMARY KEY AUTOINCREMENT,
    sales_staff_bk INTEGER NOT NULL UNIQUE,
    first_name TEXT,
    last_name TEXT,
    full_name TEXT,
    position_en TEXT,
    date_hired TEXT,
    sales_branch_code INTEGER,
    branch_city TEXT,
    branch_region TEXT,
    branch_country TEXT
);

CREATE TABLE IF NOT EXISTS dim_order_method (
    order_method_key INTEGER PRIMARY KEY AUTOINCREMENT,
    order_method_bk INTEGER NOT NULL UNIQUE,
    order_method_en TEXT
);

CREATE TABLE IF NOT EXISTS fact_order_sales (
    order_sales_key INTEGER PRIMARY KEY AUTOINCREMENT,
    order_detail_bk INTEGER NOT NULL UNIQUE,
    order_number INTEGER NOT NULL,
    date_key INTEGER NOT NULL,
    product_key INTEGER NOT NULL,
    customer_key INTEGER,
    region_key INTEGER,
    sales_staff_key INTEGER,
    order_method_key INTEGER,
    quantity INTEGER NOT NULL,
    unit_cost REAL,
    unit_price REAL,
    unit_sale_price REAL,
    revenue REAL,
    cost_amount REAL,
    gross_profit REAL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
    FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key),
    FOREIGN KEY (region_key) REFERENCES dim_region(region_key),
    FOREIGN KEY (sales_staff_key) REFERENCES dim_sales_staff(sales_staff_key),
    FOREIGN KEY (order_method_key) REFERENCES dim_order_method(order_method_key)
);

CREATE TABLE IF NOT EXISTS fact_sales_target (
    sales_target_key INTEGER PRIMARY KEY AUTOINCREMENT,
    sales_staff_bk INTEGER,
    date_key INTEGER,
    product_key INTEGER,
    retailer_code INTEGER,
    retailer_name TEXT,
    sales_target REAL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
);

CREATE TABLE IF NOT EXISTS fact_inventory_level (
    inventory_key INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key INTEGER,
    product_key INTEGER,
    inventory_count INTEGER,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
);

CREATE TABLE IF NOT EXISTS fact_product_forecast (
    forecast_key INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key INTEGER,
    product_key INTEGER,
    expected_volume INTEGER,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
);
"""


## 5. DWH resetten


In [6]:
def connect(db_path):
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA foreign_keys = ON")
    return con


def reset_dwh():
    if DWH_DB.exists():
        DWH_DB.unlink()
        log_event("INFO", "RESET_DWH", "GO_DWH", "DELETE_EXISTING_DB", 1, str(DWH_DB))

    con = connect(DWH_DB)
    con.executescript(SCHEMA_SQL)
    con.commit()
    con.close()

    log_event("INFO", "RESET_DWH", "GO_DWH", "CREATE_SCHEMA", 1, "DWH-schema aangemaakt")
    print("DWH opnieuw aangemaakt:", DWH_DB)


## 6. Brondata lezen


In [7]:
def read_sql(db_path, table_name):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', con)
    con.close()
    log_event("INFO", "EXTRACT_SQL", table_name, "READ_TABLE", len(df), db_path.name)
    return df


def read_csv_robust(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            log_event("INFO", "EXTRACT_CSV", path.name, "READ_CSV", len(df), f"encoding={enc}")
            return df
        except Exception as e:
            last_error = e

    log_event("ERROR", "EXTRACT_CSV", path.name, "FAILED", "", str(last_error))
    raise last_error


## 7. Staging bouwen


In [65]:
def build_staging():
    # Sales-bron
    order_header = read_sql(SALES_DB, "order_header")
    order_details = read_sql(SALES_DB, "order_details")
    product = read_sql(SALES_DB, "product")
    product_type = read_sql(SALES_DB, "product_type")
    product_line = read_sql(SALES_DB, "product_line")
    retailer_site = read_sql(SALES_DB, "retailer_site")
    country_sales = read_sql(SALES_DB, "country")
    order_method = read_sql(SALES_DB, "order_method")
    sales_staff = read_sql(SALES_DB, "sales_staff")
    sales_branch = read_sql(SALES_DB, "sales_branch")

    # CRM-bron
    customer = read_sql(CRM_DB, "customer")
    customer_type = read_sql(CRM_DB, "customer_type")
    customer_store = read_sql(CRM_DB, "customer_store")
    customer_hq = read_sql(CRM_DB, "customer_headquarters")
    customer_segment = read_sql(CRM_DB, "customer_segment")
    crm_country = read_sql(CRM_DB, "crm_country")
    sales_territory = read_sql(CRM_DB, "sales_territory")

    # CSV-bronnen
    inventory = read_csv_robust(INVENTORY_CSV)
    forecast = read_csv_robust(FORECAST_CSV)
    target = read_csv_robust(TARGET_CSV)

    # Dim product
    dim_product = (
        product
        .merge(product_type, on="PRODUCT_TYPE_CODE", how="left")
        .merge(product_line, on="PRODUCT_LINE_CODE", how="left")
    )

    dim_product = dim_product[[
        "PRODUCT_NUMBER", "PRODUCT_NAME", "PRODUCT_TYPE_CODE", "PRODUCT_TYPE_EN",
        "PRODUCT_LINE_CODE", "PRODUCT_LINE_EN", "INTRODUCTION_DATE", "PRODUCTION_COST", "MARGIN"
    ]].drop_duplicates("PRODUCT_NUMBER")

    dim_product = dim_product.rename(columns={
        "PRODUCT_NUMBER": "product_bk",
        "PRODUCT_NAME": "product_name",
        "PRODUCT_TYPE_CODE": "product_type_code",
        "PRODUCT_TYPE_EN": "product_type",
        "PRODUCT_LINE_CODE": "product_line_code",
        "PRODUCT_LINE_EN": "product_line",
        "INTRODUCTION_DATE": "introduction_date",
        "PRODUCTION_COST": "production_cost",
        "MARGIN": "margin"
    })

    # Dim customer
    customer_full = (
        customer_store
        .merge(customer, left_on="CUSTOMER_CODE", right_on="CUSTOMER_CODE", how="left")
        .merge(customer_type, on="CUSTOMER_TYPE_CODE", how="left")
        .merge(customer_hq[["CUSTOMER_CODEMR", "SEGMENT_CODE"]], on="CUSTOMER_CODEMR", how="left")
        .merge(customer_segment[["SEGMENT_CODE", "SEGMENT_NAME"]], on="SEGMENT_CODE", how="left")
        .merge(crm_country[["COUNTRY_CODE", "COUNTRY_EN", "SALES_TERRITORY_CODE"]], on="COUNTRY_CODE", how="left")
        .merge(sales_territory[["SALES_TERRITORY_CODE", "TERRITORY_NAME_EN"]], on="SALES_TERRITORY_CODE", how="left")
    )

    sales_sites = retailer_site.merge(
        country_sales[["COUNTRY_CODE", "COUNTRY"]],
        on="COUNTRY_CODE",
        how="left"
    )

    dim_customer_base = sales_sites.merge(
        customer_full,
        left_on="RETAILER_SITE_CODE",
        right_on="CUSTOMER_SITE_CODE",
        how="left",
        suffixes=("_sales", "_crm")
    )

    dim_customer = pd.DataFrame({
        "customer_site_bk": dim_customer_base["RETAILER_SITE_CODE"],
        "customer_bk": dim_customer_base["RETAILER_CODE"],
        "customer_master_bk": dim_customer_base["CUSTOMER_CODEMR"],
        "retailer_name": None,
        "company_name": dim_customer_base["COMPANY_NAME"],
        "customer_type": dim_customer_base["CUSTOMER_TYPE_EN"],
        "segment_code": dim_customer_base["SEGMENT_CODE"],
        "segment_name": dim_customer_base["SEGMENT_NAME"],
        "city": dim_customer_base["CITY_sales"],
        "region": dim_customer_base["REGION"],
        "postal_zone": dim_customer_base["POSTAL_ZONE"],
        "country_code": dim_customer_base["COUNTRY_CODE_sales"],
        "country": dim_customer_base["COUNTRY"],
        "territory_name": dim_customer_base["TERRITORY_NAME_EN"],
        "active_indicator": dim_customer_base["ACTIVE_INDICATOR_sales"]
    }).drop_duplicates("customer_site_bk")

    retailer_names = order_header[["RETAILER_SITE_CODE", "RETAILER_NAME"]].drop_duplicates("RETAILER_SITE_CODE")

    dim_customer = dim_customer.merge(
        retailer_names,
        left_on="customer_site_bk",
        right_on="RETAILER_SITE_CODE",
        how="left"
    )

    dim_customer["retailer_name"] = dim_customer["RETAILER_NAME"]
    dim_customer = dim_customer.drop(columns=["RETAILER_SITE_CODE", "RETAILER_NAME"])

    # Dim region
    dim_region = dim_customer[["city", "region", "country_code", "country", "territory_name"]].drop_duplicates().copy()

    dim_region["region_bk"] = (
        dim_region["city"].fillna("UNKNOWN").astype(str) + "|" +
        dim_region["region"].fillna("UNKNOWN").astype(str) + "|" +
        dim_region["country_code"].fillna(-1).astype(int).astype(str)
    )

    dim_region = dim_region[[
        "region_bk",
        "city",
        "region",
        "country_code",
        "country",
        "territory_name"
    ]]

    # Dim sales staff
    staff_full = (
        sales_staff
        .merge(sales_branch, on="SALES_BRANCH_CODE", how="left", suffixes=("_staff", "_branch"))
        .merge(country_sales[["COUNTRY_CODE", "COUNTRY"]], on="COUNTRY_CODE", how="left")
    )

    dim_sales_staff = pd.DataFrame({
        "sales_staff_bk": staff_full["SALES_STAFF_CODE"],
        "first_name": staff_full["FIRST_NAME"],
        "last_name": staff_full["LAST_NAME"],
        "full_name": staff_full["FIRST_NAME"].fillna("") + " " + staff_full["LAST_NAME"].fillna(""),
        "position_en": staff_full["POSITION_EN"],
        "date_hired": staff_full["DATE_HIRED"],
        "sales_branch_code": staff_full["SALES_BRANCH_CODE"],
        "branch_city": staff_full["CITY"],
        "branch_region": staff_full["REGION"],
        "branch_country": staff_full["COUNTRY"]
    }).drop_duplicates("sales_staff_bk")

    # Dim order method
    dim_order_method = order_method.rename(columns={
        "ORDER_METHOD_CODE": "order_method_bk",
        "ORDER_METHOD_EN": "order_method_en"
    })[[
        "order_method_bk",
        "order_method_en"
    ]].drop_duplicates("order_method_bk")

    # Fact order sales staging
    fact = order_details.merge(order_header, on="ORDER_NUMBER", how="left")

    fact["ORDER_DATE_DT"] = pd.to_datetime(
    fact["ORDER_DATE"],
    format="%d-%b-%Y %I:%M:%S %p",
    errors="coerce"
)

    missing_dates = fact[fact["ORDER_DATE_DT"].isna()]["ORDER_DATE"].drop_duplicates()

    if len(missing_dates) > 0:
        log_event(
            "WARNING",
            "BUILD_STAGING",
            "fact_order_sales",
            "MISSING_ORDER_DATES",
            len(missing_dates),
            "Ongeldige ORDER_DATE vervangen door 1900-01-01"
        )

    fact["ORDER_DATE_DT"] = fact["ORDER_DATE_DT"].fillna(pd.Timestamp("1900-01-01"))
    fact["date_key"] = fact["ORDER_DATE_DT"].dt.strftime("%Y%m%d").astype(int)

    fact["QUANTITY"] = pd.to_numeric(fact["QUANTITY"], errors="coerce").fillna(0)
    fact["UNIT_SALE_PRICE"] = pd.to_numeric(fact["UNIT_SALE_PRICE"], errors="coerce").fillna(0)
    fact["UNIT_COST"] = pd.to_numeric(fact["UNIT_COST"], errors="coerce").fillna(0)
    fact["UNIT_PRICE"] = pd.to_numeric(fact["UNIT_PRICE"], errors="coerce").fillna(0)

    fact["revenue"] = fact["QUANTITY"] * fact["UNIT_SALE_PRICE"]
    fact["cost_amount"] = fact["QUANTITY"] * fact["UNIT_COST"]
    fact["gross_profit"] = fact["revenue"] - fact["cost_amount"]

    fact_sales = fact.rename(columns={
        "ORDER_DETAIL_CODE": "order_detail_bk",
        "ORDER_NUMBER": "order_number",
        "PRODUCT_NUMBER": "product_bk",
        "RETAILER_SITE_CODE": "customer_site_bk",
        "SALES_STAFF_CODE": "sales_staff_bk",
        "ORDER_METHOD_CODE": "order_method_bk",
        "QUANTITY": "quantity",
        "UNIT_COST": "unit_cost",
        "UNIT_PRICE": "unit_price",
        "UNIT_SALE_PRICE": "unit_sale_price"
    })[[
        "order_detail_bk",
        "order_number",
        "date_key",
        "product_bk",
        "customer_site_bk",
        "sales_staff_bk",
        "order_method_bk",
        "quantity",
        "unit_cost",
        "unit_price",
        "unit_sale_price",
        "revenue",
        "cost_amount",
        "gross_profit"
    ]]

    # Dates verzamelen
    date_values = ["1900-01-01"]
    date_values.extend(fact["ORDER_DATE_DT"].dropna().dt.date.astype(str).tolist())

    for _, row in target.iterrows():
        date_values.append(f"{int(row['SALES_YEAR']):04d}-{int(row['SALES_PERIOD']):02d}-01")

    for _, row in inventory.iterrows():
        date_values.append(f"{int(row['INVENTORY_YEAR']):04d}-{int(row['INVENTORY_MONTH']):02d}-01")

    for _, row in forecast.iterrows():
        date_values.append(f"{int(row['YEAR']):04d}-{int(row['MONTH']):02d}-01")

    date_df = pd.DataFrame({"full_date": sorted(set(date_values))})

    dts = pd.to_datetime(date_df["full_date"])

    date_df["date_key"] = dts.dt.strftime("%Y%m%d").astype(int)
    date_df["day_of_month"] = dts.dt.day
    date_df["month_num"] = dts.dt.month
    date_df["month_name"] = dts.dt.month_name()
    date_df["quarter_num"] = dts.dt.quarter
    date_df["year_num"] = dts.dt.year
    date_df["week_num"] = dts.dt.isocalendar().week.astype(int)
    date_df["day_name"] = dts.dt.day_name()
    date_df["is_weekend"] = dts.dt.dayofweek.isin([5, 6]).astype(int)

    # Extra facts
    target_fact = target.copy()

    target_fact["date_key"] = target_fact.apply(
        lambda r: int(f"{int(r['SALES_YEAR']):04d}{int(r['SALES_PERIOD']):02d}01"),
        axis=1
    )

    target_fact = target_fact.rename(columns={
        "SALES_STAFF_CODE": "sales_staff_bk",
        "PRODUCT_NUMBER": "product_bk",
        "RETAILER_CODE": "retailer_code",
        "RETAILER_NAME": "retailer_name",
        "SALES_TARGET": "sales_target"
    })[[
        "sales_staff_bk",
        "date_key",
        "product_bk",
        "retailer_code",
        "retailer_name",
        "sales_target"
    ]]

    inventory_fact = inventory.copy()

    inventory_fact["date_key"] = inventory_fact.apply(
        lambda r: int(f"{int(r['INVENTORY_YEAR']):04d}{int(r['INVENTORY_MONTH']):02d}01"),
        axis=1
    )

    inventory_fact = inventory_fact.rename(columns={
        "PRODUCT_NUMBER": "product_bk",
        "INVENTORY_COUNT": "inventory_count"
    })[[
        "date_key",
        "product_bk",
        "inventory_count"
    ]]

    forecast_fact = forecast.copy()

    forecast_fact["date_key"] = forecast_fact.apply(
        lambda r: int(f"{int(r['YEAR']):04d}{int(r['MONTH']):02d}01"),
        axis=1
    )

    forecast_fact = forecast_fact.rename(columns={
        "PRODUCT_NUMBER": "product_bk",
        "EXPECTED_VOLUME": "expected_volume"
    })[[
        "date_key",
        "product_bk",
        "expected_volume"
    ]]

    staging = {
        "dim_date": date_df,
        "dim_product": dim_product,
        "dim_customer": dim_customer,
        "dim_region": dim_region,
        "dim_sales_staff": dim_sales_staff,
        "dim_order_method": dim_order_method,
        "fact_order_sales": fact_sales,
        "fact_sales_target": target_fact,
        "fact_inventory_level": inventory_fact,
        "fact_product_forecast": forecast_fact
    }

    for name, df_stage in staging.items():
        log_event("INFO", "STAGING", name, "BUILD", len(df_stage), "Staging dataframe opgebouwd")

    return staging

## 8. Dimensies en feiten laden


In [72]:
def insert_dataframe(con, table, df):
    df.to_sql(table, con, if_exists="append", index=False)
    log_event("INFO", "LOAD_TABLE", table, "INSERT", len(df), "Data geladen")


def load_dimensions(con, staging):
    insert_dataframe(con, "dim_date", staging["dim_date"])

    unknown_product = pd.DataFrame([{
        "product_bk": -1,
        "product_name": "Unknown Product",
        "product_type_code": None,
        "product_type": "Unknown",
        "product_line_code": None,
        "product_line": "Unknown",
        "introduction_date": None,
        "production_cost": None,
        "margin": None
    }])

    dim_product = staging["dim_product"].copy()
    dim_product["product_bk"] = pd.to_numeric(
        dim_product["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    dim_product_with_unknown = pd.concat(
        [unknown_product, dim_product],
        ignore_index=True
    ).drop_duplicates(subset=["product_bk"])

    insert_dataframe(con, "dim_product", dim_product_with_unknown)

    insert_dataframe(con, "dim_customer", staging["dim_customer"])
    insert_dataframe(con, "dim_region", staging["dim_region"])
    insert_dataframe(con, "dim_sales_staff", staging["dim_sales_staff"])
    insert_dataframe(con, "dim_order_method", staging["dim_order_method"])


def load_facts(con, staging):
    product_lookup = pd.read_sql_query(
        "SELECT product_key, product_bk FROM dim_product", con
    )

    product_lookup["product_bk"] = pd.to_numeric(
        product_lookup["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    product_lookup = product_lookup.set_index("product_bk")["product_key"].to_dict()

    customer_lookup_df = pd.read_sql_query(
    "SELECT customer_key, customer_site_bk FROM dim_customer", con
    )

    customer_lookup_df["customer_site_bk"] = pd.to_numeric(
    customer_lookup_df["customer_site_bk"],
    errors="coerce"
    ).astype("Int64")

    customer_lookup = customer_lookup_df.set_index("customer_site_bk")["customer_key"].to_dict()

    staff_lookup = pd.read_sql_query(
        "SELECT sales_staff_key, sales_staff_bk FROM dim_sales_staff", con
    ).set_index("sales_staff_bk")["sales_staff_key"].to_dict()

    method_lookup = pd.read_sql_query(
        "SELECT order_method_key, order_method_bk FROM dim_order_method", con
    ).set_index("order_method_bk")["order_method_key"].to_dict()

    region_lookup = pd.read_sql_query(
        "SELECT region_key, region_bk FROM dim_region", con
    ).set_index("region_bk")["region_key"].to_dict()

    unknown_product_key = pd.read_sql_query(
        "SELECT product_key FROM dim_product WHERE product_bk = -1", con
    )["product_key"].iloc[0]

    cust_region = pd.read_sql_query(
        "SELECT customer_site_bk, city, region, country_code FROM dim_customer", con
    )

    cust_region["region_bk"] = (
        cust_region["city"].fillna("UNKNOWN").astype(str) + "|" +
        cust_region["region"].fillna("UNKNOWN").astype(str) + "|" +
        cust_region["country_code"].fillna(-1).astype(int).astype(str)
    )

    cust_region_lookup = cust_region.set_index("customer_site_bk")["region_bk"].to_dict()

    # fact_order_sales
    fact = staging["fact_order_sales"].copy()

    fact["product_bk"] = pd.to_numeric(
        fact["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    fact["product_key"] = fact["product_bk"].map(product_lookup)
   
    fact["customer_site_bk"] = pd.to_numeric(
    fact["customer_site_bk"],
    errors="coerce"
    ).astype("Int64")
    
    fact["customer_key"] = fact["customer_site_bk"].map(customer_lookup)
    fact["sales_staff_key"] = fact["sales_staff_bk"].map(staff_lookup)
    fact["order_method_key"] = fact["order_method_bk"].map(method_lookup)
    fact["region_key"] = fact["customer_site_bk"].map(cust_region_lookup).map(region_lookup)

    missing_products = fact[fact["product_key"].isna()]["product_bk"].drop_duplicates()

    if len(missing_products) > 0:
        log_event(
            "WARNING",
            "LOAD_FACTS",
            "fact_order_sales",
            "MISSING_PRODUCT_KEYS",
            len(missing_products),
            "Ontbrekende product_bk gemapt naar Unknown Product"
        )

    fact["product_key"] = fact["product_key"].fillna(unknown_product_key)

    fact_final = fact[[
        "order_detail_bk",
        "order_number",
        "date_key",
        "product_key",
        "customer_key",
        "region_key",
        "sales_staff_key",
        "order_method_key",
        "quantity",
        "unit_cost",
        "unit_price",
        "unit_sale_price",
        "revenue",
        "cost_amount",
        "gross_profit"
    ]]

    insert_dataframe(con, "fact_order_sales", fact_final)

    # fact_sales_target
    target = staging["fact_sales_target"].copy()

    target["product_bk"] = pd.to_numeric(
        target["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    target["product_key"] = target["product_bk"].map(product_lookup)

    missing_target_products = target[target["product_key"].isna()]["product_bk"].drop_duplicates()

    if len(missing_target_products) > 0:
        log_event(
            "WARNING",
            "LOAD_FACTS",
            "fact_sales_target",
            "MISSING_PRODUCT_KEYS",
            len(missing_target_products),
            "Ontbrekende product_bk gemapt naar Unknown Product"
        )

    target["product_key"] = target["product_key"].fillna(unknown_product_key)

    target_final = target[[
        "sales_staff_bk",
        "date_key",
        "product_key",
        "retailer_code",
        "retailer_name",
        "sales_target"
    ]]

    insert_dataframe(con, "fact_sales_target", target_final)

    # fact_inventory_level
    inventory = staging["fact_inventory_level"].copy()

    inventory["product_bk"] = pd.to_numeric(
        inventory["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    inventory["product_key"] = inventory["product_bk"].map(product_lookup)

    missing_inventory_products = inventory[inventory["product_key"].isna()]["product_bk"].drop_duplicates()

    if len(missing_inventory_products) > 0:
        log_event(
            "WARNING",
            "LOAD_FACTS",
            "fact_inventory_level",
            "MISSING_PRODUCT_KEYS",
            len(missing_inventory_products),
            "Ontbrekende product_bk gemapt naar Unknown Product"
        )

    inventory["product_key"] = inventory["product_key"].fillna(unknown_product_key)

    inventory_final = inventory[[
        "date_key",
        "product_key",
        "inventory_count"
    ]]

    insert_dataframe(con, "fact_inventory_level", inventory_final)

    # fact_product_forecast
    forecast = staging["fact_product_forecast"].copy()

    forecast["product_bk"] = pd.to_numeric(
        forecast["product_bk"],
        errors="coerce"
    ).fillna(-1).astype(int)

    forecast["product_key"] = forecast["product_bk"].map(product_lookup)

    missing_forecast_products = forecast[forecast["product_key"].isna()]["product_bk"].drop_duplicates()

    if len(missing_forecast_products) > 0:
        log_event(
            "WARNING",
            "LOAD_FACTS",
            "fact_product_forecast",
            "MISSING_PRODUCT_KEYS",
            len(missing_forecast_products),
            "Ontbrekende product_bk gemapt naar Unknown Product"
        )

    forecast["product_key"] = forecast["product_key"].fillna(unknown_product_key)

    forecast_final = forecast[[
        "date_key",
        "product_key",
        "expected_volume"
    ]]

    insert_dataframe(con, "fact_product_forecast", forecast_final)

## 9. Analyseviews maken


In [73]:
ANALYSIS_VIEWS_SQL = '''
DROP VIEW IF EXISTS vw_omzet_per_maand_product;
CREATE VIEW vw_omzet_per_maand_product AS
SELECT
    d.year_num,
    d.month_num,
    d.month_name,
    p.product_name,
    p.product_type,
    p.product_line,
    SUM(f.quantity) AS totaal_aantal,
    SUM(f.revenue) AS omzet,
    SUM(f.gross_profit) AS brutowinst
FROM fact_order_sales f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_product p ON f.product_key = p.product_key
GROUP BY d.year_num, d.month_num, d.month_name, p.product_name, p.product_type, p.product_line;

DROP VIEW IF EXISTS vw_omzet_per_regio;
CREATE VIEW vw_omzet_per_regio AS
SELECT
    r.country,
    r.territory_name,
    r.region,
    r.city,
    SUM(f.quantity) AS totaal_aantal,
    SUM(f.revenue) AS omzet,
    SUM(f.gross_profit) AS brutowinst
FROM fact_order_sales f
LEFT JOIN dim_region r ON f.region_key = r.region_key
GROUP BY r.country, r.territory_name, r.region, r.city;

DROP VIEW IF EXISTS vw_top_klanten;
CREATE VIEW vw_top_klanten AS
SELECT
    c.customer_site_bk,
    c.retailer_name,
    c.company_name,
    c.customer_type,
    c.segment_name,
    c.country,
    c.region,
    c.city,
    COUNT(DISTINCT f.order_number) AS aantal_orders,
    SUM(f.quantity) AS totaal_aantal,
    SUM(f.revenue) AS omzet,
    SUM(f.gross_profit) AS brutowinst
FROM fact_order_sales f
LEFT JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.customer_site_bk, c.retailer_name, c.company_name, c.customer_type, c.segment_name, c.country, c.region, c.city;

DROP VIEW IF EXISTS vw_product_combinations;
CREATE VIEW vw_product_combinations AS
SELECT
    f1.order_number,
    p1.product_name AS product_1,
    p2.product_name AS product_2,
    COUNT(*) AS combinatie_count
FROM fact_order_sales f1
JOIN fact_order_sales f2
    ON f1.order_number = f2.order_number
   AND f1.product_key < f2.product_key
JOIN dim_product p1 ON f1.product_key = p1.product_key
JOIN dim_product p2 ON f2.product_key = p2.product_key
GROUP BY f1.order_number, p1.product_name, p2.product_name;

DROP VIEW IF EXISTS vw_product_combinations_summary;
CREATE VIEW vw_product_combinations_summary AS
SELECT
    product_1,
    product_2,
    COUNT(*) AS aantal_samen_besteld
FROM vw_product_combinations
GROUP BY product_1, product_2;
'''


def create_analysis_views(con):
    con.executescript(ANALYSIS_VIEWS_SQL)
    con.commit()
    log_event("INFO", "CREATE_VIEWS", "analysis_views", "CREATE", 5, "Analyseviews aangemaakt")


## 10. Pipeline uitvoeren


In [74]:
def run_dwh_pipeline():
    log_event("INFO", "RUN_DWH_PIPELINE", "", "START", "", "Start DWH ETL")

    reset_dwh()
    staging = build_staging()

    con = connect(DWH_DB)
    load_dimensions(con, staging)
    load_facts(con, staging)
    create_analysis_views(con)

    summary = []
    for table in [
        "dim_date", "dim_product", "dim_customer", "dim_region", "dim_sales_staff", "dim_order_method",
        "fact_order_sales", "fact_sales_target", "fact_inventory_level", "fact_product_forecast"
    ]:
        count = pd.read_sql_query(f"SELECT COUNT(*) AS aantal FROM {table}", con)["aantal"].iloc[0]
        summary.append({"tabel": table, "aantal_rijen": int(count)})
        log_event("INFO", "ROW_COUNT", table, "COUNT_ROWS", int(count), "DWH rijtelling")

    con.close()

    log_event("INFO", "RUN_DWH_PIPELINE", "", "END", "", "DWH ETL afgerond")
    return pd.DataFrame(summary)


reset_dwh_log()
check_files()
summary = run_dwh_pipeline()
summary

Logbestand gereset: /Users/cmok/Documents/GitHub/DEAI-SE4/Great_Outdoors/logs/go_dwh_etl.log
Gevonden: CRM-data.sqlite
Gevonden: GO_SALES-data.sqlite
Gevonden: GO_STAFF-data.sqlite
Gevonden: INVENTORY_LEVELS-data.csv
Gevonden: PRODUCT_FORECAST-data.csv
Gevonden: SALES_TARGET-data.csv
Alle bronbestanden zijn gevonden.
DWH opnieuw aangemaakt: /Users/cmok/Documents/GitHub/DEAI-SE4/Great_Outdoors/GO_DWH.db


,tabel,aantal_rijen
0,dim_date,929
1,dim_product,116
2,dim_customer,391
3,dim_region,237
4,dim_sales_staff,96
5,dim_order_method,7
6,fact_order_sales,40990
7,fact_sales_target,39530
8,fact_inventory_level,3888
9,fact_product_forecast,3872


## 11. Voorbeeldanalyses


In [75]:
def q(sql):
    con = connect(DWH_DB)
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


In [76]:
# Top 10 klanten op omzet
q('''
SELECT retailer_name, company_name, country, region, city, omzet, aantal_orders
FROM vw_top_klanten
ORDER BY omzet DESC
LIMIT 10
''')


,retailer_name,company_name,country,region,city,omzet,aantal_orders
0,Esportes Grumari,Esportes Grumari,Brazil,SP,São Paulo,6100030.98,15
1,Falcon Outfitters,Falcon Outfitters,Canada,Québec,Montréal,2444242.26,23
2,Falcon Outfitters,Falcon Outfitters,Canada,British Columbia,Dawson Creek,1949006.06,21
3,Falcon Outfitters,Falcon Outfitters,Canada,Ontario,Ottawa,1813769.82,22
4,Falcon Outfitters,Falcon Outfitters,Canada,Nova Scotia,Halifax,1592542.10,22
5,Falcon Outfitters,Falcon Outfitters,Canada,Ontario,Toronto,1484220.94,22
6,In die Berge!,In die Berge!,Germany,Thüringen,Erfurt,1481756.44,21
7,Ultra Sports,Ultra Sports,Canada,Saskatchewan,Regina,1399551.98,15
8,In die Berge!,In die Berge!,Germany,Baden-Württemberg,Freiburg,1291343.84,21
9,Ultra Sports,Ultra Sports,Canada,Ontario,Sudbury,1261003.94,15


In [77]:
# Top 10 producten op omzet
q('''
SELECT product_name, product_type, product_line, SUM(omzet) AS omzet, SUM(totaal_aantal) AS aantal
FROM vw_omzet_per_maand_product
GROUP BY product_name, product_type, product_line
ORDER BY omzet DESC
LIMIT 10
''')


,product_name,product_type,product_line,omzet,aantal
0,Star Dome,Tents,Camping Equipment,14948640.60,25564
1,Star Gazer 3,Tents,Camping Equipment,13044951.40,22660
2,Star Gazer 2,Tents,Camping Equipment,9123447.66,18262
3,Star Gazer 6,Tents,Camping Equipment,7219022.62,10504
4,Hailstorm Titanium Woods Set,Woods,Golf Equipment,5251573.20,4890
5,Hibernator Extreme,Sleeping Bags,Camping Equipment,4715059.40,20596
6,Husky Rope 100,Rope,Mountaineering Equipment,4613953.30,14500
7,Canyon Mule Extreme Backpack,Packs,Camping Equipment,4335772.24,11052
8,Mountain Man Extreme,Watches,Personal Accessories,4057957.42,17456
9,Canyon Mule Journey Backpack,Packs,Camping Equipment,3622382.68,11598


In [78]:
# Omzet per regio
q('''
SELECT country, territory_name, region, city, omzet
FROM vw_omzet_per_regio
ORDER BY omzet DESC
LIMIT 10
''')


,country,territory_name,region,city,omzet
0,Brazil,Americas,SP,São Paulo,6100030.98
1,Canada,Americas,Québec,Montréal,5008053.66
2,United Kingdom,Central Europe,NaN,London,4931603.46
3,Germany,Central Europe,Hessen,Frankfurt,4436798.74
4,Canada,Americas,Ontario,Toronto,3953999.96
5,Canada,Americas,British Columbia,Vancouver,2570220.68
6,Netherlands,NaN,NaN,Amsterdam,2502831.82
7,Canada,Americas,Ontario,Ottawa,2364833.38
8,France,Central Europe,NaN,Paris,2251488.98
9,Canada,Americas,British Columbia,Dawson Creek,1949006.06


In [79]:
# Productcombinaties die het vaakst samen besteld worden
q('''
SELECT product_1, product_2, aantal_samen_besteld
FROM vw_product_combinations_summary
ORDER BY aantal_samen_besteld DESC
LIMIT 10
''')


,product_1,product_2,aantal_samen_besteld
0,Granite Axe,Granite Extreme,156
1,Firefly Climbing Lamp,Granite Extreme,139
2,Granite Hammer,Granite Grip,138
3,Firefly Climbing Lamp,Granite Axe,136
4,Granite Grip,Granite Extreme,132
5,Granite Grip,Granite Axe,130
6,Granite Hammer,Granite Extreme,130
7,Firefly Charger,Granite Extreme,129
8,Firefly Charger,Granite Axe,127
9,Firefly Climbing Lamp,Firefly Charger,127


## 12. Uitleg voor assessment

Dit DWH is gebouwd voor het knelpunt dat Great Outdoors onvoldoende inzicht heeft in bestelgedrag.

De belangrijkste feitentabel is `fact_order_sales`.  
De granulariteit hiervan is **één orderregel per rij**. Daardoor kan het bedrijf analyseren:

- welke producten worden besteld;
- hoeveel producten worden besteld;
- hoeveel omzet wordt gemaakt;
- welke klanten bestellen;
- in welke regio’s omzet ontstaat;
- in welke periodes producten worden besteld;
- welke productcombinaties vaak samen voorkomen.

De belangrijkste dimensies zijn:
- `dim_date` voor tijdsanalyses;
- `dim_product` voor productanalyses;
- `dim_customer` voor klant- en segmentanalyses;
- `dim_region` voor regionale analyses;
- `dim_sales_staff` voor verkoopmedewerkeranalyses;
- `dim_order_method` voor bestelmethode-analyses.

Daarnaast zijn analyseviews toegevoegd voor:
- omzet per maand en product;
- omzet per regio;
- topklanten;
- productcombinaties.
